# Data Setup

This is the first notebook for my AI in Healthcare project. The goal here is simple — download the NHANES 2017-2018 health survey data, merge the files together, create a label column, handle missing values, and save everything to Google Drive.

I'm using the 2017-2018 cycle because it's the last complete pre-COVID NHANES cycle. After 2018, data collection was disrupted and CDC had to merge cycles together which makes things messier.

In [1]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install pyreadstat -q

import pandas as pd
import numpy as np
import os

print("done")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.5 MB/s eta 0:00:00
done


In [2]:
# Paths
base = '/content/drive/MyDrive/PTSD_Project'

raw = base + '/data/raw'
processed = base + '/data/processed'

print("paths set")

paths set


## Downloading NHANES data files

NHANES splits data by topic into separate files. I need 4 of them:
- DPQ_J: Depression screener (PHQ-9 questions) — this gives me my target label
- DEMO_J: Demographics — age, gender, income, education, marital status
- ALQ_J: Alcohol use — drinking frequency and amount
- SLQ_J: Sleep — hours of sleep and sleep disorder history

Each file is in .XPT format (SAS format used by CDC). All files are connected by a unique patient ID called SEQN.

In [3]:
# Download NHANES files
import urllib.request

files = {
    'DPQ_J.XPT':  'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DPQ_J.XPT',
    'DEMO_J.XPT': 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.XPT',
    'ALQ_J.XPT':  'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/ALQ_J.XPT',
    'SLQ_J.XPT':  'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/SLQ_J.XPT',
}

for filename, url in files.items():
    save_path = raw + '/' + filename
    urllib.request.urlretrieve(url, save_path)
    print(filename, "downloaded")

print("\nall files saved to drive")

DPQ_J.XPT downloaded
DEMO_J.XPT downloaded
ALQ_J.XPT downloaded
SLQ_J.XPT downloaded

all files saved to drive


In [4]:
# Read files
dpq  = pd.read_sas(raw + '/DPQ_J.XPT',  format='xport')
demo = pd.read_sas(raw + '/DEMO_J.XPT', format='xport')
alq  = pd.read_sas(raw + '/ALQ_J.XPT',  format='xport')
slq  = pd.read_sas(raw + '/SLQ_J.XPT',  format='xport')

print("DPQ  shape:", dpq.shape)
print("DEMO shape:", demo.shape)
print("ALQ  shape:", alq.shape)
print("SLQ  shape:", slq.shape)

DPQ  shape: (5533, 11)
DEMO shape: (9254, 46)
ALQ  shape: (5533, 10)
SLQ  shape: (6161, 11)


## Selecting relevant columns

Each file has way more columns than I need. I'm keeping only the ones that make sense for predicting PTSD risk — the 9 PHQ depression questions, basic demographics, alcohol use, and sleep hours.

In [5]:
# Select columns
dpq_cols  = ['SEQN', 'DPQ010', 'DPQ020', 'DPQ030', 'DPQ040',
             'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090']

demo_cols = ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1',
             'DMDEDUC2', 'INDFMPIR', 'DMDMARTL']

alq_cols  = ['SEQN', 'ALQ111', 'ALQ121', 'ALQ130']

slq_cols  = ['SEQN', 'SLD012', 'SLQ050', 'SLQ120']

dpq  = dpq[dpq_cols]
demo = demo[demo_cols]
alq  = alq[alq_cols]
slq  = slq[slq_cols]

print("DPQ:", dpq.shape)
print("DEMO:", demo.shape)
print("ALQ:", alq.shape)
print("SLQ:", slq.shape)

DPQ: (5533, 10)
DEMO: (9254, 7)
ALQ: (5533, 4)
SLQ: (6161, 4)


## Merging all 4 files into one table

Using inner join on SEQN so I only keep patients who appear in all 4 files. This drops people who didn't complete all parts of the survey.

In [6]:
# Merge all files on SEQN
df = dpq.merge(demo, on='SEQN', how='inner') \
        .merge(alq,  on='SEQN', how='inner') \
        .merge(slq,  on='SEQN', how='inner')

print("merged shape:", df.shape)
print("\ncolumns:", df.columns.tolist())

merged shape: (5533, 22)

columns: ['SEQN', 'DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'DMDMARTL', 'ALQ111', 'ALQ121', 'ALQ130', 'SLD012', 'SLQ050', 'SLQ120']


## Creating the PHQ total score and target label

PHQ-9 is a standard depression screening tool. Each of the 9 questions is scored 0-3. The total score ranges from 0-27.

I'm using PHQ total score >= 10 as the high risk label. This is the clinically validated cutoff where doctors start considering treatment for moderate depression. Since NHANES doesn't have a direct PTSD diagnosis, this PHQ-based proxy is a reasonable approximation — I'll state this as a limitation in the report.

One issue I ran into: the SAS file format stores zeros as a very small float (~5.39e-79). I fixed this by only keeping values that are actually 0, 1, 2, or 3 — anything else gets dropped.

In [7]:
# Create PHQ score and label
phq_cols = ['DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050',
            'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090']

# force convert to numeric, then round to nearest integer
for col in phq_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].round().astype('Int64')  # Int64 supports NaN
    # anything not 0,1,2,3 is invalid - set to NaN
    df[col] = df[col].where(df[col].isin([0, 1, 2, 3]), other=np.nan)

# drop rows where any PHQ question is missing
df = df.dropna(subset=phq_cols)

# convert to regular int now
df[phq_cols] = df[phq_cols].astype(int)

# total PHQ score
df['PHQ_total'] = df[phq_cols].sum(axis=1)

# label: 1 = high risk (PHQ >= 10), 0 = low risk
df['label'] = (df['PHQ_total'] >= 10).astype(int)

print("shape:", df.shape)
print("\nPHQ sample values:")
print(df[phq_cols].head())
print("\nlabel distribution:")
print(df['label'].value_counts())

shape: (5068, 24)

PHQ sample values:
   DPQ010  DPQ020  DPQ030  DPQ040  DPQ050  DPQ060  DPQ070  DPQ080  DPQ090
0       0       0       0       0       0       0       0       0       0
1       0       0       0       0       0       0       0       0       0
2       0       0       0       0       0       0       0       0       0
4       1       0       1       0       0       0       0       0       0
5       0       0       1       0       0       0       0       0       0

label distribution:
label
0    4609
1     459
Name: count, dtype: int64


In [8]:
# fix remaining SAS near-zero values in non-PHQ columns
other_cols = ['ALQ111', 'ALQ121', 'ALQ130', 'SLD012', 'SLQ050', 'SLQ120',
              'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'DMDMARTL']

for col in other_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # if absolute value is less than 1e-10, it's a fake zero
    df[col] = df[col].apply(lambda x: 0 if pd.notnull(x) and abs(x) < 1e-10 else x)

# also fix PHQ_total
df['PHQ_total'] = df['PHQ_total'].apply(lambda x: 0 if pd.notnull(x) and abs(x) < 1e-10 else x)

print("sample after fix:")
print(df[['SLQ120', 'ALQ111', 'PHQ_total']].head(10))

sample after fix:
    SLQ120  ALQ111  PHQ_total
0      0.0     1.0          0
1      1.0     2.0          0
2      2.0     2.0          0
4      3.0     1.0          2
5      2.0     1.0          1
6      2.0     1.0          8
7      3.0     1.0          2
8      4.0     1.0          4
9      3.0     1.0          2
10     1.0     1.0          5


In [9]:
def fix_sas_zeros(val):
    try:
        if abs(float(val)) < 1e-10:
            return 0
        return val
    except:
        return val

df = df.applymap(fix_sas_zeros)

print("sample check:")
print(df[['SLQ120', 'ALQ121', 'ALQ130', 'PHQ_total']].head())
print("\nany weird values left?")
print((df < 1e-10).sum().sum())

sample check:
   SLQ120  ALQ121  ALQ130  PHQ_total
0     0.0     7.0     1.0          0
1     1.0     NaN     NaN          0
2     2.0     NaN     NaN          0
4     3.0     5.0     1.0          2
5     2.0     0.0     NaN          1

any weird values left?
43167


/tmp/ipykernel_882/4228880961.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(fix_sas_zeros)


## Handling missing values

Different strategies for different columns:
- ALQ121 and ALQ130 (drinking frequency/amount): missing here means the person doesn't drink, so I fill with 0
- ALQ111 (ever drank?): genuinely missing for ~7% of people, easier to just drop those rows
- INDFMPIR (income ratio) and SLD012 (sleep hours): continuous columns, median impute
- DMDEDUC2 and DMDMARTL (education, marital status): categorical, mode impute

In [10]:
# Handle missing values

# ALQ121 and ALQ130 missing means person doesn't drink, fill with 0
df['ALQ121'] = df['ALQ121'].fillna(0)
df['ALQ130'] = df['ALQ130'].fillna(0)

# drop rows where ALQ111 is missing (only 7%, cleaner to just drop)
df = df.dropna(subset=['ALQ111'])

# median impute for continuous columns
df['INDFMPIR'] = df['INDFMPIR'].fillna(df['INDFMPIR'].median())
df['SLD012']   = df['SLD012'].fillna(df['SLD012'].median())

# mode impute for categorical columns
df['DMDEDUC2'] = df['DMDEDUC2'].fillna(df['DMDEDUC2'].mode()[0])
df['DMDMARTL'] = df['DMDMARTL'].fillna(df['DMDMARTL'].mode()[0])

print("shape after handling missing:", df.shape)
print("\nany missing values left?", df.isnull().sum().sum())

shape after handling missing: (5065, 24)

any missing values left? 0


In [11]:
# Save to Drive
df.to_csv(processed + '/merged_clean.csv', index=False)
print("saved to drive")
print("file:", processed + '/merged_clean.csv')
print("shape:", df.shape)

saved to drive
file: /content/drive/MyDrive/PTSD_Project/data/processed/merged_clean.csv
shape: (5065, 24)
